In [10]:
#!/usr/bin/env python3
"""
scripts/process_local_drc.py
Diagnostic Parser for DRC COUSP 2026 Data
"""

import os
import csv
import re
from pathlib import Path
from datetime import datetime

# ==========================================================
# PATH CONFIGURATION
# ==========================================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

PROJECT_ROOT = SCRIPT_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "incomming"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "ebola_drc.csv"

LOCAL_DOWNLOAD_PATH = Path("/Users/koissi/Downloads/drc_ebola_cases_consolidated (1).csv")

print("=" * 70)
print("PROCESSING DOWNLOADED DRC COUSP 2026 REPOSITORY (DIAGNOSTIC PASS)")
print("=" * 70)

if not LOCAL_DOWNLOAD_PATH.exists():
    print(f"❌ Error: Cannot find file at {LOCAL_DOWNLOAD_PATH}")
    exit(1)

daily_aggregation = {}

print("-> Ingesting and decoding file layout...")
with open(LOCAL_DOWNLOAD_PATH, mode="r", encoding="utf-8-sig", errors="ignore") as f:
    lines = f.readlines()

print(f"   Total lines read from file: {len(lines)}")

# --- DIAGNOSTIC PRINT: Show exactly what the first 5 lines look like ---
print("\n--- DIAGNOSTIC SNAPSHOT (FIRST 5 VALID LINES) ---")
printed_count = 0
for idx, l in enumerate(lines[:10]):
    if l.strip() and "sep=" not in l.lower():
        parts = re.split(r'[\t;,]', l.strip())
        print(f"Line {idx} Split Parts: {parts[:6]}")
        printed_count += 1
        if printed_count >= 5:
            break
print("-" * 50 + "\n")

# Process lines manually
for line_idx, raw_line in enumerate(lines):
    line = raw_line.strip()
    if not line or line_idx == 0 or "sep=" in line.lower():
        continue  
        
    parts = re.split(r'[\t;,]', line)
    parts = [p.strip().strip('"\'') for p in parts if p is not None]
    
    if len(parts) < 2:
        continue
        
    # Attempt to locate date string anywhere in the first 3 columns
    date_iso = None
    for part in parts[:3]:
        # Strip trailing times if present (e.g., "2026-05-24 00:00:00")
        clean_part = part.split()[0] if " " in part else part
        
        # Test common formats
        for fmt in ("%Y-%m-%d", "%m/%d/%y", "%m/%d/%Y", "%d/%m/%Y", "%Y/%m/%d"):
            try:
                date_iso = datetime.strptime(clean_part, fmt).strftime("%Y-%m-%d")
                break
            except ValueError:
                continue
        if date_iso:
            break

    # EMERGENCY FALLBACK: If date parsing completely fails, we'll assign a placeholder 
    # based on a 2026 context sequence so your script doesn't drop the data while we debug.
    if not date_iso:
        date_iso = "2026-06-23" 

    def clean_num(val_str):
        if not val_str or val_str.upper() in ("NA", "NAN", ""):
            return 0.0
        cleaned = re.sub(r'[^\d.]', '', val_str)
        try:
            return float(cleaned) if cleaned else 0.0
        except ValueError:
            return 0.0

    try:
        suspected = clean_num(parts[2]) if len(parts) > 2 else 0.0
        confirmed = clean_num(parts[3]) if len(parts) > 3 else 0.0
        deaths = clean_num(parts[4]) if len(parts) > 4 else 0.0
        recoveries = clean_num(parts[5]) if len(parts) > 5 else 0.0
    except IndexError:
        continue

    if date_iso not in daily_aggregation:
        daily_aggregation[date_iso] = {
            "date": date_iso,
            "country": "DR Congo",
            "suspected_cases": suspected,
            "confirmed_cases": confirmed,
            "deaths": deaths,
            "recoveries": recoveries
        }
    else:
        rec = daily_aggregation[date_iso]
        rec["suspected_cases"] += suspected
        rec["confirmed_cases"] += confirmed
        rec["deaths"] += deaths
        rec["recoveries"] += recoveries

# ==========================================================
# PREPARE DATASET ROWS & SAVE
# ==========================================================
final_dataset = []
for date_key in sorted(daily_aggregation.keys(), reverse=True):
    record = daily_aggregation[date_key]
    
    c = record["confirmed_cases"]
    d = record["deaths"]
    record["case_fatality_rate_pct"] = round((d / c) * 100, 2) if c > 0 else 0.0
    
    record["suspected_cases"] = int(record["suspected_cases"])
    record["confirmed_cases"] = int(record["confirmed_cases"])
    record["deaths"] = int(record["deaths"])
    record["recoveries"] = int(record["recoveries"])
    record["source_url"] = "COUSP-RDC Local Plain Text Parser"
    
    final_dataset.append(record)

FIELDS = [
    "date", "country", "suspected_cases", "confirmed_cases", 
    "deaths", "recoveries", "case_fatality_rate_pct", "source_url"
]

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(final_dataset)

print("=" * 70)
print("LOCAL METRIC HARMONIZATION COMPLETED")
print("=" * 70)
print(f"✔ File output: data/incomming/{OUTPUT_PATH.name}")
print(f"✔ Total Chronological Days Saved: {len(final_dataset)}")
print("=" * 70)

PROCESSING DOWNLOADED DRC COUSP 2026 REPOSITORY (DIAGNOSTIC PASS)
-> Ingesting and decoding file layout...
   Total lines read from file: 1898

--- DIAGNOSTIC SNAPSHOT (FIRST 5 VALID LINES) ---
Line 0 Split Parts: ['location_country', 'location_level', 'location_name', 'location_name_source', 'location_code', 'location_code_type']
Line 1 Split Parts: ['COD', '3', 'Mongbalu', 'Mongbalu', 'CD540510', 'pcode']
Line 2 Split Parts: ['COD', '3', 'Mongbalu', 'Mongbalu', 'CD540510', 'pcode']
Line 3 Split Parts: ['COD', '3', 'Mongbalu', 'Mongbalu', 'CD540510', 'pcode']
Line 4 Split Parts: ['COD', '3', 'Bunia', 'Bunia', 'CD540202', 'pcode']
--------------------------------------------------

LOCAL METRIC HARMONIZATION COMPLETED
✔ File output: data/incomming/ebola_drc.csv
✔ Total Chronological Days Saved: 1


In [13]:
#!/usr/bin/env python3
"""
scripts/process_local_drc.py
Long-to-Wide Pivot Aggregator for INRB-UMIE 2026 Data Structure
"""

import os
import csv
from pathlib import Path
from datetime import datetime

# ==========================================================
# PATH CONFIGURATION
# ==========================================================
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    SCRIPT_DIR = Path.cwd()

PROJECT_ROOT = SCRIPT_DIR.parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "incomming"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = OUTPUT_DIR / "ebola_drc.csv"

LOCAL_DOWNLOAD_PATH = Path("/Users/koissi/Downloads/drc_ebola_cases_consolidated (1).csv")

print("=" * 70)
print("PROCESSING INRB-UMIE LONG SCHEMA DATASET")
print("=" * 70)

if not LOCAL_DOWNLOAD_PATH.exists():
    print(f"❌ Error: Cannot find file at {LOCAL_DOWNLOAD_PATH}")
    exit(1)

# Grouping map: { "YYYY-MM-DD": { metrics } }
daily_aggregation = {}

print("-> Ingesting and pivoting rows on relational attributes...")
with open(LOCAL_DOWNLOAD_PATH, mode="r", encoding="utf-8-sig") as f:
    # Handle auto-detection of delimiters (Tabs vs Commas)
    sample = f.read(2048)
    f.seek(0)
    dialect = csv.excel_tab if "\t" in sample else csv.excel
    
    reader = csv.DictReader(f, dialect=dialect)

    for row in reader:
        # 1. Date normalization (mapping from 'reference_date')
        raw_date = row.get("reference_date")
        if not raw_date:
            continue
            
        try:
            date_iso = datetime.strptime(raw_date.strip(), "%m/%d/%y").strftime("%Y-%m-%d")
        except ValueError:
            try:
                date_iso = datetime.strptime(raw_date.strip(), "%Y-%m-%d").strftime("%Y-%m-%d")
            except ValueError:
                continue

        # Initialize the composite dictionary day slot if empty
        if date_iso not in daily_aggregation:
            daily_aggregation[date_iso] = {
                "date": date_iso,
                "country": "DR Congo",
                "suspected_cases": 0,
                "confirmed_cases": 0,
                "deaths": 0,
                "recoveries": 0
            }

        day_ref = daily_aggregation[date_iso]

        # 2. Extract value column safely
        try:
            numeric_val = int(float(row.get("value", 0) or 0))
        except ValueError:
            numeric_val = 0

        # 3. Pivot conditions using the long formatting combinations
        measure = str(row.get("measure", "")).lower().strip()
        classification = str(row.get("case_classification", "")).lower().strip()

        if measure == "cases":
            if classification == "confirmed":
                day_ref["confirmed_cases"] += numeric_val
            elif classification == "suspected":
                day_ref["suspected_cases"] += numeric_val
                
        elif measure == "deaths":
            # Captures deaths recorded across classification labels
            day_ref["deaths"] += numeric_val
            
        elif measure in ("recoveries", "recovered", "gueris"):
            day_ref["recoveries"] += numeric_val

# ==========================================================
# CALCULATION & SAVING TRANSFORMATION PASS
# ==========================================================
final_dataset = []
for date_key in sorted(daily_aggregation.keys(), reverse=True):
    record = daily_aggregation[date_key]
    
    c = record["confirmed_cases"]
    d = record["deaths"]
    
    # Calculate Case Fatality Rate safely
    record["case_fatality_rate_pct"] = round((d / c) * 100, 2) if c > 0 else 0.0
    record["source_url"] = "INRB-UMIE 2026 Live Dashboard Repository"
    
    final_dataset.append(record)

FIELDS = [
    "date", "country", "suspected_cases", "confirmed_cases", 
    "deaths", "recoveries", "case_fatality_rate_pct", "source_url"
]

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(final_dataset)

print("\n" + "=" * 70)
print("LOCAL METRIC HARMONIZATION COMPLETED")
print("=" * 70)
print(f"✔ Transformed Target File: data/incomming/{OUTPUT_PATH.name}")
print(f"✔ Unified Dynamic Chronological Records Compiled: {len(final_dataset)}")
print("=" * 70)

PROCESSING INRB-UMIE LONG SCHEMA DATASET
-> Ingesting and pivoting rows on relational attributes...

LOCAL METRIC HARMONIZATION COMPLETED
✔ Transformed Target File: data/incomming/ebola_drc.csv
✔ Unified Dynamic Chronological Records Compiled: 34
